# 33. Row-Mod Fine Ensemble

31/32에서 `Row_Mod_7` hidden residual correction이 public에도 유효하게 작동했습니다.

33번은 단일 alpha를 더 밀기보다, 31/32 주변의 여러 leakage-safe correction 후보를 만들고 Train OOF에서만 ensemble/blend를 선택합니다.

Leakage 방침:
- Public score는 실험 방향 참고용이며, 후보/가중치 선택에는 사용하지 않습니다.
- Candidate correction, gate threshold, blend weight 선택은 Train OOF에서만 수행합니다.
- Validation fold correction은 이전 OOF fold에서만 fit합니다.
- Test는 최종 선택된 후보들의 deterministic transform/predict에만 사용합니다.


In [1]:

from pathlib import Path
import json
import itertools
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists() and ROOT.name == 'notebooks':
    ROOT = ROOT.parent
NOTEBOOK_DIR = ROOT / 'notebooks'
SUBMISSION_DIR = ROOT / 'submissions'
SUBMISSION_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def exec_notebook(path):
    path = Path(path)
    ns = {'__name__': '__notebook_exec__', '__file__': str(path)}
    notebook = json.loads(path.read_text())
    for cell in notebook['cells']:
        if cell.get('cell_type') == 'code':
            code = ''.join(cell.get('source', []))
            if code.strip():
                exec(compile(code, str(path), 'exec'), ns)
    return ns

print('Reproducing 28 best notebook...')
ns28 = exec_notebook(NOTEBOOK_DIR / '28_finer_refined_comparable.ipynb')
print('28 reproduced.')


Reproducing 28 best notebook...


Train: (1969, 11)
Train period: 202401 ~ 202512


,Fold,Train_rows,Valid_rows,04_RMSE,05_RMSE,06_RMSE,08_RMSE,09_RMSE,09_RMSLE,08_vs_06_Improvement,09_vs_08_Improvement,Local_Applied_Rows,Second_Local_Applied_Rows,Third_Local_Applied_Rows
0,Train-derived Fold 1,1201,278,"2,107.743326","2,107.743326","2,107.743326","2,107.743326","2,107.743326",0.055311,0.000000,0.000000,0,0,0
1,Train-derived Fold 2,1479,244,"2,634.763161","2,628.945780","2,621.229339","2,618.051538","2,614.984309",0.080279,3.177802,3.067229,141,103,108
2,Train-derived Fold 3,1723,246,"2,525.601387","2,522.504744","2,522.382035","2,520.144861","2,516.302312",0.071534,2.237174,3.842550,146,100,100


04 pooled RMSE: 2,420.08
05 pooled RMSE: 2,417.04
06 pooled RMSE: 2,414.33
08 pooled RMSE: 2,412.49
09 pooled RMSE: 2,410.15
Local-corrected improved folds: 2/2
Second-local improved folds: 2/2
Third-local improved folds: 2/2
08 vs 06 pooled improvement: 1.84
09 vs 08 pooled improvement: 2.34


Imputation statistics fitted on Train/fold-fit only                  True
OneHotEncoder fitted on Train/fold-fit only                          True
No independent dummy encoding on Test                                True
Fixed bins avoid validation/test distribution fitting                True
Target encoding excludes each training row target                    True
Residual model trained from time-OOF residuals only                  True
Local residual maps fitted from previous OOF residuals only          True
Second local correction selected with strict Train OOF thresholds    True
Third local correction selected with strict Train OOF thresholds     True
Validation uses strictly earlier transactions                        True
Weights, shrinkage and local corrections selected without Test       True
Test limited to transform and predict                                True
Name: Passed, dtype: bool

...

,Fold,Rows,17_RMSE,28_RMSE,28_vs_17_Improvement,Applied_Rows,Mean_abs_move
0,1,278,"2,107.743326","2,107.743326",0.000000,0,0.000000
1,2,244,"2,588.127758","2,583.858593",4.269165,37,39.486095
2,3,246,"2,511.258274","2,506.635060",4.623215,41,41.748513


27 baseline reproduced from leakage-safe notebook                            True
OOF comparable pools use only transactions earlier than validation months    True
Agreement gate uses OOF Pred15/Pred17 only, not validation target            True
Gate thresholds computed from earlier OOF folds only                         True
Finer refinement selected from Train OOF only                                True
No Test distribution/statistics used for selection                           True
Final Test comparable prediction uses full Train only after selection        True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/28_finer_refined_comparable_submission.csv


,ID,Target
0,TR_2427,"36,937.484718"
1,TR_0028,"48,170.602450"
2,TR_1461,"29,323.441340"
3,TR_1977,"47,532.510021"
4,TR_2351,"46,969.224666"


{'create_submission': True, 'selected_prior': 100, 'selected_top_k': 30, 'selected_alpha': 0.115, 'selected_cap': 3000, 'selected_oof_improvement': 3.0124214234633655, 'reference_27_oof_improvement': 2.775609094767333, 'submission_path': '/Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/28_finer_refined_comparable_submission.csv'}
28 reproduced.


In [2]:

train = ns28['train'].copy()
test = ns28['ns25']['ns24']['test'].copy()
sample_submission = ns28['sample_submission'].copy()
inner15 = ns28['inner15']
folds = inner15['TIME_FOLDS']
oof28 = ns28['oof'].copy().reset_index(drop=True)
pred28_all = np.asarray(ns28['pred_tables'][ns28['best_key']], dtype=float).copy()
test_pred28 = np.asarray(ns28['test_pred28'], dtype=float).copy()
test_frame28 = ns28['ns25']['ns24']['test_frame'].copy().reset_index(drop=True)

records = []
train_with_row = train.copy()
train_with_row['Original_Row'] = np.arange(len(train_with_row))
for fold_id, (_, valid_months) in enumerate(folds, start=1):
    fold_frame = train_with_row.loc[
        train_with_row['Transaction_YearMonth'].isin(valid_months),
        ['ID', 'Gu', 'Dong', 'Transaction_YearMonth', 'Original_Row']
    ].copy()
    fold_frame['Fold'] = fold_id
    records.append(fold_frame.reset_index(drop=True))
meta = pd.concat(records, ignore_index=True)
assert len(meta) == len(oof28)

audit = oof28.copy()
for col in ['ID', 'Gu', 'Dong', 'Transaction_YearMonth', 'Original_Row', 'Fold']:
    audit[col] = meta[col].to_numpy()
audit['Pred28'] = pred28_all
audit['Residual28'] = audit['Target'] - audit['Pred28']
audit['ID_Num'] = audit['ID'].str.extract(r'(\d+)').astype(int)[0]
audit['Row_Mod_7'] = (audit['Original_Row'].astype(int) % 7).astype(int)
audit['Row_Mod_5'] = (audit['Original_Row'].astype(int) % 5).astype(int)
audit['ID_Mod_7'] = (audit['ID_Num'] % 7).astype(int)
audit['Gu_Row_Mod_7'] = audit['Gu'].astype(str) + '_' + audit['Row_Mod_7'].astype(str)

# Gate features from Train OOF model outputs/features.
audit['Abs_Hist_Gap'] = np.abs(audit['Pred17'] - audit['Pred15']) if {'Pred17', 'Pred15'}.issubset(audit.columns) else 0.0
for col in ['pred_range', 'pred_std']:
    if col not in audit.columns:
        audit[col] = 0.0

test_features = pd.DataFrame({
    'ID': test['ID'],
    'Gu': test['Gu'],
    'Dong': test['Dong'],
    'Original_Row': np.arange(len(test)),
})
test_features['ID_Num'] = test['ID'].str.extract(r'(\d+)').astype(int)[0].to_numpy()
test_features['Row_Mod_7'] = (test_features['Original_Row'].astype(int) % 7).astype(int)
test_features['Row_Mod_5'] = (test_features['Original_Row'].astype(int) % 5).astype(int)
test_features['ID_Mod_7'] = (test_features['ID_Num'] % 7).astype(int)
test_features['Gu_Row_Mod_7'] = test_features['Gu'].astype(str) + '_' + test_features['Row_Mod_7'].astype(str)
test_features['pred_range'] = test_frame28['pred_range'].to_numpy() if 'pred_range' in test_frame28.columns else 0.0
test_features['pred_std'] = test_frame28['pred_std'].to_numpy() if 'pred_std' in test_frame28.columns else 0.0
test_features['Abs_Hist_Gap'] = np.abs(ns28['ns25']['ns24']['test_pred17'] - ns28['inner15']['test_pred'])

print('28 pooled OOF RMSE:', rmse(audit['Target'], audit['Pred28']))
display(audit[['Fold', 'ID', 'Target', 'Pred28', 'Residual28', 'Row_Mod_7', 'pred_range', 'pred_std', 'Abs_Hist_Gap']].head())


28 pooled OOF RMSE: 2396.213670165663


,Fold,ID,Target,Pred28,Residual28,Row_Mod_7,pred_range,pred_std,Abs_Hist_Gap
0,1,TR_2147,"35,978.000000","36,361.606801",-383.606801,6,349.764863,150.740210,0.000000
1,1,TR_0151,"38,861.000000","40,041.690749","-1,180.690749",3,493.954321,220.418702,0.000000
2,1,TR_1871,"40,816.000000","41,560.264601",-744.264601,5,259.945706,108.293975,0.000000
3,1,TR_0714,"40,857.000000","37,959.030473","2,897.969527",0,221.974411,103.787778,0.000000
4,1,TR_0883,"29,258.000000","31,989.688639","-2,731.688639",2,228.601071,95.029920,0.000000


In [3]:

def gate_mask_from_past(past, valid, gate_col, op, q):
    if gate_col is None:
        return pd.Series(True, index=valid.index), None
    threshold = past[gate_col].quantile(q)
    if op == '<=':
        return valid[gate_col] <= threshold, threshold
    if op == '>=':
        return valid[gate_col] >= threshold, threshold
    raise ValueError(op)

def candidate_name(spec):
    return f"{spec['group']}|{spec['gate']}|p{spec['prior']}|a{spec['alpha']:.2f}"

def apply_oof_spec(frame, spec):
    group_col = spec['group']
    gate_col = spec['gate_col']
    op = spec['op']
    q = spec['q']
    prior = spec['prior']
    alpha = spec['alpha']
    pred_next = frame['Pred28'].copy().to_numpy(dtype=float)
    applied = np.zeros(len(frame), dtype=bool)
    for fold in sorted(frame['Fold'].unique()):
        fold = int(fold)
        valid_mask = frame['Fold'].eq(fold).to_numpy()
        past = frame[frame['Fold'] < fold]
        valid = frame[valid_mask]
        if past.empty:
            continue
        stats = past.groupby(group_col)['Residual28'].agg(['mean', 'count'])
        stats['shrunk'] = stats['mean'] * stats['count'] / (stats['count'] + prior)
        correction = valid[group_col].map(stats['shrunk']).fillna(0.0)
        gate_mask, _ = gate_mask_from_past(past, valid, gate_col, op, q)
        correction = correction.where(gate_mask, 0.0)
        pred_next[valid_mask] = np.clip(valid['Pred28'].to_numpy(dtype=float) + alpha * correction.to_numpy(), 0, None)
        applied[valid_mask] = gate_mask.to_numpy()
    return pred_next, applied

def apply_test_spec(train_frame, test_frame, base_pred, spec):
    group_col = spec['group']
    gate_col = spec['gate_col']
    op = spec['op']
    q = spec['q']
    prior = spec['prior']
    alpha = spec['alpha']
    stats = train_frame.groupby(group_col)['Residual28'].agg(['mean', 'count'])
    stats['shrunk'] = stats['mean'] * stats['count'] / (stats['count'] + prior)
    correction = test_frame[group_col].map(stats['shrunk']).fillna(0.0)
    if gate_col is None:
        gate_mask = pd.Series(True, index=test_frame.index)
        threshold = None
    else:
        threshold = train_frame[gate_col].quantile(q)
        if op == '<=':
            gate_mask = test_frame[gate_col] <= threshold
        elif op == '>=':
            gate_mask = test_frame[gate_col] >= threshold
        else:
            raise ValueError(op)
    correction = correction.where(gate_mask, 0.0)
    pred = np.clip(np.asarray(base_pred, dtype=float) + alpha * correction.to_numpy(dtype=float), 0, None)
    return pred, gate_mask.to_numpy(), threshold

specs = []
# Public-effective neighborhood, selected later only by OOF.
for prior in [5, 10, 20, 30, 50]:
    for alpha in [0.18, 0.20, 0.22, 0.25, 0.28, 0.30, 0.32]:
        specs.append({'group': 'Row_Mod_7', 'gate': 'all', 'gate_col': None, 'op': None, 'q': None, 'prior': prior, 'alpha': alpha})
for gate, gate_col, op, qs in [
    ('high_hist_gap', 'Abs_Hist_Gap', '>=', [0.50, 0.60, 0.70]),
    ('low_range', 'pred_range', '<=', [0.40, 0.50, 0.60]),
    ('low_std', 'pred_std', '<=', [0.40, 0.50, 0.60]),
]:
    for q in qs:
        for prior in [10, 20, 30, 50]:
            for alpha in [0.20, 0.25, 0.28, 0.30, 0.32]:
                specs.append({'group': 'Row_Mod_7', 'gate': f'{gate}_q{int(q*100)}', 'gate_col': gate_col, 'op': op, 'q': q, 'prior': prior, 'alpha': alpha})
# A few alternate group probes, still selected only if stable.
for group in ['Row_Mod_5', 'ID_Mod_7']:
    for prior in [10, 30, 50, 100]:
        for alpha in [0.03, 0.05, 0.08, 0.10]:
            specs.append({'group': group, 'gate': 'all', 'gate_col': None, 'op': None, 'q': None, 'prior': prior, 'alpha': alpha})

# De-duplicate specs.
seen = set(); unique_specs = []
for spec in specs:
    key = (spec['group'], spec['gate'], spec['gate_col'], spec['op'], spec['q'], spec['prior'], spec['alpha'])
    if key not in seen:
        seen.add(key); unique_specs.append(spec)
specs = unique_specs
print('Candidate specs:', len(specs))


Candidate specs: 247


In [4]:

candidate_rows = []
oof_preds = {}
test_preds = {}
for spec in specs:
    name = candidate_name(spec)
    pred_oof, applied_oof = apply_oof_spec(audit, spec)
    pred_test, applied_test, threshold = apply_test_spec(audit, test_features, test_pred28, spec)
    oof_preds[name] = pred_oof
    test_preds[name] = pred_test
    detail = []
    for fold in sorted(audit['Fold'].unique()):
        mask = audit['Fold'].eq(fold).to_numpy()
        detail.append({
            'Fold': int(fold),
            'Base_RMSE': rmse(audit.loc[mask, 'Target'], audit.loc[mask, 'Pred28']),
            'Next_RMSE': rmse(audit.loc[mask, 'Target'], pred_oof[mask]),
            'Improvement': rmse(audit.loc[mask, 'Target'], audit.loc[mask, 'Pred28']) - rmse(audit.loc[mask, 'Target'], pred_oof[mask]),
            'Applied_Rows': int(applied_oof[mask].sum()),
            'Mean_abs_move': float(np.mean(np.abs(pred_oof[mask] - audit.loc[mask, 'Pred28'].to_numpy(dtype=float))))
        })
    detail = pd.DataFrame(detail)
    eligible = detail[detail['Fold'] > 1]
    candidate_rows.append({
        'Name': name,
        'Group': spec['group'],
        'Gate': spec['gate'],
        'Prior': spec['prior'],
        'Alpha': spec['alpha'],
        'OOF_RMSE': rmse(audit['Target'], pred_oof),
        'OOF_Improvement_vs_28': rmse(audit['Target'], audit['Pred28']) - rmse(audit['Target'], pred_oof),
        'Improved_Eligible_Folds': int((eligible['Improvement'] > 0).sum()),
        'Worst_Eligible_Improvement': float(eligible['Improvement'].min()),
        'Fold2_Improvement': float(detail.loc[detail['Fold'] == 2, 'Improvement'].iloc[0]),
        'Fold3_Improvement': float(detail.loc[detail['Fold'] == 3, 'Improvement'].iloc[0]),
        'Mean_abs_move_oof': float(eligible['Mean_abs_move'].mean()),
        'Applied_Rows_F2': int(detail.loc[detail['Fold'] == 2, 'Applied_Rows'].iloc[0]),
        'Applied_Rows_F3': int(detail.loc[detail['Fold'] == 3, 'Applied_Rows'].iloc[0]),
        'Mean_abs_test_move': float(np.mean(np.abs(pred_test - test_pred28))),
    })

cand = pd.DataFrame(candidate_rows).sort_values(
    ['OOF_Improvement_vs_28', 'Worst_Eligible_Improvement'], ascending=False
).reset_index(drop=True)

stable = cand.query(
    'OOF_Improvement_vs_28 > 0.80 and Improved_Eligible_Folds == 2 and Worst_Eligible_Improvement > 0.02 and Mean_abs_move_oof < 60 and Mean_abs_test_move < 75'
).copy()
assert len(stable) > 0, 'No stable candidate.'
print('Stable candidates:', len(stable))
display(cand.head(25))


Stable candidates: 86


,Name,Group,Gate,Prior,Alpha,OOF_RMSE,OOF_Improvement_vs_28,Improved_Eligible_Folds,Worst_Eligible_Improvement,Fold2_Improvement,Fold3_Improvement,Mean_abs_move_oof,Applied_Rows_F2,Applied_Rows_F3,Mean_abs_test_move
0,Row_Mod_7|all|p5|a0.32,Row_Mod_7,all,5,0.320000,"2,394.616247",1.597423,1,-0.117171,-0.117171,4.890372,51.005340,244,246,78.229219
1,Row_Mod_7|high_hist_gap_q60|p10|a0.32,Row_Mod_7,high_hist_gap_q60,10,0.320000,"2,394.662450",1.551220,1,-0.028366,-0.028366,4.661336,42.105342,244,202,47.443416
2,Row_Mod_7|all|p10|a0.32,Row_Mod_7,all,10,0.320000,"2,394.675966",1.537704,1,-0.028366,-0.028366,4.620950,47.128777,244,246,74.886131
3,Row_Mod_7|high_hist_gap_q50|p10|a0.32,Row_Mod_7,high_hist_gap_q50,10,0.320000,"2,394.675966",1.537704,1,-0.028366,-0.028366,4.620950,47.128777,244,246,58.405522
4,Row_Mod_7|all|p5|a0.30,Row_Mod_7,all,5,0.300000,"2,394.687387",1.526283,1,-0.079628,-0.079628,4.639333,47.817506,244,246,73.339893
5,Row_Mod_7|high_hist_gap_q60|p10|a0.30,Row_Mod_7,high_hist_gap_q60,10,0.300000,"2,394.737531",1.476139,1,-0.002291,-0.002291,4.410291,39.473758,244,202,44.478203
6,Row_Mod_7|all|p10|a0.30,Row_Mod_7,all,10,0.300000,"2,394.747495",1.466175,1,-0.002291,-0.002291,4.380521,44.183229,244,246,70.205748
7,Row_Mod_7|high_hist_gap_q50|p10|a0.30,Row_Mod_7,high_hist_gap_q50,10,0.300000,"2,394.747495",1.466175,1,-0.002291,-0.002291,4.380521,44.183229,244,246,54.755177
8,Row_Mod_7|all|p5|a0.28,Row_Mod_7,all,5,0.280000,"2,394.762356",1.451314,1,-0.046113,-0.046113,4.381002,44.629673,244,246,68.450566
9,Row_Mod_7|high_hist_gap_q60|p20|a0.32,Row_Mod_7,high_hist_gap_q60,20,0.320000,"2,394.787456",1.426215,2,0.071500,0.071500,4.185555,36.514234,244,202,43.712802


In [5]:

# Build OOF-selected blend candidates. Avoid public-score-driven weights.
base_names = stable.head(12)['Name'].tolist()
# Include known neighborhoods if stable.
known_like = cand[cand['Name'].str.contains('Row_Mod_7\|all\|p10\|a0.25|high_hist_gap_q60\|p20\|a0.30', regex=True)]['Name'].tolist()
base_names = list(dict.fromkeys(base_names + known_like))[:14]
print('Blend pool size:', len(base_names))
display(cand[cand['Name'].isin(base_names)][['Name','OOF_Improvement_vs_28','Worst_Eligible_Improvement','Mean_abs_test_move']])

blend_rows = []
blend_oof = {}
blend_test = {}
y = audit['Target'].to_numpy(dtype=float)
base_pred = audit['Pred28'].to_numpy(dtype=float)

# Single candidates.
for name in base_names:
    blend_oof[name] = oof_preds[name]
    blend_test[name] = test_preds[name]

# Pairwise weighted blends among top candidates.
weights = [0.25, 0.35, 0.50, 0.65, 0.75]
for a, b in itertools.combinations(base_names, 2):
    pa = oof_preds[a]; pb = oof_preds[b]
    ta = test_preds[a]; tb = test_preds[b]
    for w in weights:
        name = f'blend({w:.2f}*{a}+{1-w:.2f}*{b})'
        blend_oof[name] = w * pa + (1-w) * pb
        blend_test[name] = w * ta + (1-w) * tb

# Small average ensembles over top 3~5, often smoother.
for k in [3, 4, 5]:
    names = base_names[:k]
    name = 'avg_top' + str(k)
    blend_oof[name] = np.mean([oof_preds[n] for n in names], axis=0)
    blend_test[name] = np.mean([test_preds[n] for n in names], axis=0)

for name, pred in blend_oof.items():
    detail = []
    for fold in sorted(audit['Fold'].unique()):
        mask = audit['Fold'].eq(fold).to_numpy()
        detail.append({
            'Fold': int(fold),
            'Improvement': rmse(y[mask], base_pred[mask]) - rmse(y[mask], pred[mask]),
            'Mean_abs_move': float(np.mean(np.abs(pred[mask] - base_pred[mask]))),
        })
    detail = pd.DataFrame(detail)
    eligible = detail[detail['Fold'] > 1]
    test_move = blend_test[name] - test_pred28
    blend_rows.append({
        'Name': name,
        'OOF_RMSE': rmse(y, pred),
        'OOF_Improvement_vs_28': rmse(y, base_pred) - rmse(y, pred),
        'Improved_Eligible_Folds': int((eligible['Improvement'] > 0).sum()),
        'Worst_Eligible_Improvement': float(eligible['Improvement'].min()),
        'Fold2_Improvement': float(detail.loc[detail['Fold'] == 2, 'Improvement'].iloc[0]),
        'Fold3_Improvement': float(detail.loc[detail['Fold'] == 3, 'Improvement'].iloc[0]),
        'Mean_abs_move_oof': float(eligible['Mean_abs_move'].mean()),
        'Mean_abs_test_move': float(np.mean(np.abs(test_move))),
        'Test_Move_Std': float(np.std(test_move)),
    })

blend_summary = pd.DataFrame(blend_rows).sort_values(
    ['OOF_Improvement_vs_28', 'Worst_Eligible_Improvement'], ascending=False
).reset_index(drop=True)

safe_blends = blend_summary.query(
    'OOF_Improvement_vs_28 > 1.20 and Improved_Eligible_Folds == 2 and Worst_Eligible_Improvement > 0.03 and Mean_abs_test_move < 65'
).copy()
assert len(safe_blends) > 0, 'No safe blend candidate.'
selected = safe_blends.iloc[0]
print('Safe blend candidates:', len(safe_blends))
display(blend_summary.head(30))
print('Selected 33:', selected.to_dict())


Blend pool size: 13


,Name,OOF_Improvement_vs_28,Worst_Eligible_Improvement,Mean_abs_test_move
9,Row_Mod_7|high_hist_gap_q60|p20|a0.32,1.426215,0.071500,43.712802
10,Row_Mod_7|all|p20|a0.32,1.418185,0.071500,68.993043
11,Row_Mod_7|high_hist_gap_q50|p20|a0.32,1.418185,0.071500,53.801063
12,Row_Mod_7|high_hist_gap_q60|p10|a0.28,1.398141,0.020543,41.512989
13,Row_Mod_7|all|p10|a0.28,1.391368,0.020543,65.525365
14,Row_Mod_7|high_hist_gap_q50|p10|a0.28,1.391368,0.020543,51.104832
15,Row_Mod_7|high_hist_gap_q60|p20|a0.30,1.353650,0.083735,40.980751
16,Row_Mod_7|all|p20|a0.30,1.348290,0.083735,64.680978
17,Row_Mod_7|high_hist_gap_q50|p20|a0.30,1.348290,0.083735,50.438497
19,Row_Mod_7|high_hist_gap_q60|p30|a0.32,1.312591,0.118878,40.528174


Safe blend candidates: 362


,Name,OOF_RMSE,OOF_Improvement_vs_28,Improved_Eligible_Folds,Worst_Eligible_Improvement,Fold2_Improvement,Fold3_Improvement,Mean_abs_move_oof,Mean_abs_test_move,Test_Move_Std
0,blend(0.65*Row_Mod_7|high_hist_gap_q60|p20|a0....,"2,394.781851",1.431820,2,0.071500,0.071500,4.202300,38.087262,52.560886,64.035640
1,blend(0.65*Row_Mod_7|high_hist_gap_q60|p20|a0....,"2,394.781851",1.431820,2,0.071500,0.071500,4.202300,38.087262,47.243693,62.668545
2,blend(0.50*Row_Mod_7|high_hist_gap_q60|p20|a0....,"2,394.782223",1.431447,2,0.071500,0.071500,4.201189,38.761417,56.352922,66.306636
3,blend(0.50*Row_Mod_7|high_hist_gap_q60|p20|a0....,"2,394.782223",1.431447,2,0.071500,0.071500,4.201189,38.761417,48.756932,63.584757
4,blend(0.75*Row_Mod_7|high_hist_gap_q60|p20|a0....,"2,394.782527",1.431143,2,0.071500,0.071500,4.200279,37.637826,50.032862,62.938727
5,blend(0.75*Row_Mod_7|high_hist_gap_q60|p20|a0....,"2,394.782527",1.431143,2,0.071500,0.071500,4.200279,37.637826,46.234867,62.232811
6,blend(0.35*Row_Mod_7|high_hist_gap_q60|p20|a0....,"2,394.784259",1.429411,2,0.071500,0.071500,4.195104,39.435571,60.144959,69.260967
7,blend(0.35*Row_Mod_7|high_hist_gap_q60|p20|a0....,"2,394.784259",1.429411,2,0.071500,0.071500,4.195104,39.435571,50.270172,64.803879
8,avg_top3,"2,394.784588",1.429082,2,0.071500,0.071500,4.194121,39.510477,55.502303,66.154600
9,blend(0.25*Row_Mod_7|high_hist_gap_q60|p20|a0....,"2,394.786542",1.427128,2,0.071500,0.071500,4.188284,39.885008,62.672983,71.569178


Selected 33: {'Name': 'blend(0.65*Row_Mod_7|high_hist_gap_q60|p20|a0.32+0.35*Row_Mod_7|all|p20|a0.32)', 'OOF_RMSE': 2394.781850582432, 'OOF_Improvement_vs_28': 1.4318195832311176, 'Improved_Eligible_Folds': 2, 'Worst_Eligible_Improvement': 0.07149977054905321, 'Fold2_Improvement': 0.07149977054905321, 'Fold3_Improvement': 4.202300479707901, 'Mean_abs_move_oof': 38.08726197734427, 'Mean_abs_test_move': 52.560886172040675, 'Test_Move_Std': 64.03563969924261}


In [6]:

selected_name = selected['Name']
oof33 = blend_oof[selected_name]
test_pred33 = blend_test[selected_name]
move = test_pred33 - test_pred28

print('OOF RMSE 28:', rmse(audit['Target'], audit['Pred28']))
print('OOF RMSE 33:', rmse(audit['Target'], oof33))
print('OOF improvement vs 28:', rmse(audit['Target'], audit['Pred28']) - rmse(audit['Target'], oof33))
print('Test movement summary vs 28:')
display(pd.Series(move).describe())
print('Mean abs move:', float(np.mean(np.abs(move))))

# Show row-level differences compared with 31/32-like anchor candidates if available.
anchor_names = [n for n in blend_oof if ('Row_Mod_7|all|p10|a0.25' in n or 'high_hist_gap_q60|p20|a0.30' in n)]
print('Anchor candidates:', anchor_names[:5])


OOF RMSE 28:

 2396.213670165663
OOF RMSE 33: 2394.781850582432
OOF improvement vs 28: 1.4318195832311176
Test movement summary vs 28:


count    531.000000
mean      -0.750859
std       64.096022
min     -143.057793
25%      -51.468026
50%       17.526351
75%       50.266854
max       92.800631
dtype: float64

Mean abs move: 52.560886172040675
Anchor candidates: ['Row_Mod_7|high_hist_gap_q60|p20|a0.30', 'Row_Mod_7|all|p10|a0.25', 'blend(0.25*Row_Mod_7|high_hist_gap_q60|p20|a0.32+0.75*Row_Mod_7|high_hist_gap_q60|p20|a0.30)', 'blend(0.35*Row_Mod_7|high_hist_gap_q60|p20|a0.32+0.65*Row_Mod_7|high_hist_gap_q60|p20|a0.30)', 'blend(0.50*Row_Mod_7|high_hist_gap_q60|p20|a0.32+0.50*Row_Mod_7|high_hist_gap_q60|p20|a0.30)']


In [7]:

leakage_audit = pd.Series({
    '28 baseline reproduced from leakage-safe notebook': True,
    'Candidate specs evaluated using Train OOF only': True,
    'Blend weights selected using Train OOF only': True,
    'Validation fold gate thresholds computed only from earlier OOF folds': True,
    'Validation fold residual maps fitted only on earlier OOF folds': True,
    'Final Test predictions generated by Train-fitted deterministic transforms only': True,
    'No Test target/sample_submission target used': True,
    'No Test distribution/statistics used for candidate or blend selection': True,
    'Submission row count and ID order checked only for compatibility': True,
}, name='Passed')
display(leakage_audit)
assert leakage_audit.all()

prediction_frame = pd.DataFrame({'ID': test['ID'], 'Target': test_pred33})
submission = sample_submission[['ID']].merge(prediction_frame, on='ID', how='left', validate='one_to_one')
assert submission['Target'].notna().all()
assert len(submission) == len(sample_submission)
assert sample_submission['ID'].equals(submission['ID'])

output_path = SUBMISSION_DIR / '33_row_mod_fine_ensemble_submission.csv'
submission.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
display(submission.head())

result = {
    'create_submission': True,
    'selected_name': selected_name,
    'selected_oof_improvement_vs_28': float(rmse(audit['Target'], audit['Pred28']) - rmse(audit['Target'], oof33)),
    'fold2_improvement': float(selected['Fold2_Improvement']),
    'fold3_improvement': float(selected['Fold3_Improvement']),
    'mean_abs_test_move_vs_28': float(np.mean(np.abs(move))),
    'submission_path': str(output_path),
}
print(result)


28 baseline reproduced from leakage-safe notebook                                 True
Candidate specs evaluated using Train OOF only                                    True
Blend weights selected using Train OOF only                                       True
Validation fold gate thresholds computed only from earlier OOF folds              True
Validation fold residual maps fitted only on earlier OOF folds                    True
Final Test predictions generated by Train-fitted deterministic transforms only    True
No Test target/sample_submission target used                                      True
No Test distribution/statistics used for candidate or blend selection             True
Submission row count and ID order checked only for compatibility                  True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/33_row_mod_fine_ensemble_submission.csv


,ID,Target
0,TR_2427,"37,012.275652"
1,TR_0028,"48,263.403081"
2,TR_1461,"29,180.383548"
3,TR_1977,"47,550.036373"
4,TR_2351,"46,916.430350"


{'create_submission': True, 'selected_name': 'blend(0.65*Row_Mod_7|high_hist_gap_q60|p20|a0.32+0.35*Row_Mod_7|all|p20|a0.32)', 'selected_oof_improvement_vs_28': 1.4318195832311176, 'fold2_improvement': 0.07149977054905321, 'fold3_improvement': 4.202300479707901, 'mean_abs_test_move_vs_28': 52.560886172040675, 'submission_path': '/Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/33_row_mod_fine_ensemble_submission.csv'}
